This script filters the ATKIS land use dataset to residential and mixed-use settlement areas and exports the selected polygons as a GeoPackage for further analysis.

In [ ]:
import geopandas as gpd
from pathlib import Path
from datetime import datetime

# =============================================================================
# CONFIGURATION
# =============================================================================

ATKIS_SHP = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\ATKIS\ATKIS\bkg_shape_712\sie02_f.shp"
OUTPUT_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\ATKIS\ATKIS_41001_41006.gpkg"

RELEVANT_TYPES = [
    "AX_Wohnbauflaeche",
    "AX_FlaecheGemischterNutzung"
]

# =============================================================================
# MAIN
# =============================================================================

def main():
    print("\n" + "=" * 80)
    print("STEP 1: ATKIS FILTER AND EXPORT")
    print("=" * 80)

    if not Path(ATKIS_SHP).exists():
        print(f"File not found: {ATKIS_SHP}")
        return

    start = datetime.now()
    print(f"Start: {start.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Load data
    print(f"Loading {Path(ATKIS_SHP).name}...")
    gdf = gpd.read_file(ATKIS_SHP)
    print(f"→ {len(gdf):,} features | CRS: {gdf.crs}")

    # Display all available OBJART_TXT values
    if "OBJART_TXT" in gdf.columns:
        print("\nAvailable OBJART_TXT values:")
        for val, count in gdf["OBJART_TXT"].value_counts().items():
            marker = "*" if val in RELEVANT_TYPES else " "
            print(f"  {marker} {val}: {count:,}")
    else:
        print("Column 'OBJART_TXT' not found.")
        print(f"Available columns: {list(gdf.columns)}")
        return

    # Filter relevant land-use types
    print("\nFiltering relevant land-use types...")
    gdf_filtered = gdf[gdf["OBJART_TXT"].isin(RELEVANT_TYPES)].copy()
    gdf_filtered = gdf_filtered.reset_index(drop=True)

    if len(gdf_filtered) == 0:
        print("No matching features found.")
        return

    # Summary statistics
    print("\nResults after filtering:")
    print("=" * 60)

    for val in RELEVANT_TYPES:
        subset = gdf_filtered[gdf_filtered["OBJART_TXT"] == val]
        count = len(subset)

        if count > 0:
            area_km2 = subset.geometry.area.sum() / 1e6
            mean_area_ha = subset.geometry.area.mean() / 1e4

            print(f"  {val}:")
            print(f"    Number of features: {count:,}")
            print(f"    Total area:         {area_km2:.2f} km²")
            print(f"    Mean area:          {mean_area_ha:.2f} ha")
        else:
            print(f"  {val}: 0 features")

    total_area = gdf_filtered.geometry.area.sum() / 1e6
    print("=" * 60)
    print(f"  TOTAL: {len(gdf_filtered):,} polygons | {total_area:.2f} km²")

    # Save output
    output_path = Path(OUTPUT_GPKG)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    gdf_filtered.to_file(
        output_path,
        driver="GPKG",
        layer="settlement_areas"
    )

    size_mb = output_path.stat().st_size / 1024 / 1024

    print(f"\nSaved: {output_path.name} ({size_mb:.1f} MB)")
    print("Layer: 'settlement_areas'")
    print(f"Path:  {output_path}")
    print(f"\nDuration: {datetime.now() - start}")
    print("Done.\n")


if __name__ == "__main__":
    main()